# C5 · Vision-language models on video: people and cars (15 min)

**Question:** what can image-language models tell us about a video when they only see frames, and how far can we trust them?

Flow: keyframes → detector + tracker baseline (counts over time, unique people and cars) → search the video timeline with sentences (CLIP) → ask the VLM about single frames and check against the detector → one multi-frame question about the whole clip → live panels.

Data: three short street videos from the Intel IoT DevKit sample videos (CC BY 4.0): `people`, `cars`, `street` (people, bicycles and cars).
Outputs go to `outputs/c5/`.

In [ ]:
# Make the project folder importable (works from notebooks/ or the project root)
import sys, pathlib
ROOT = pathlib.Path.cwd().resolve()
ROOT = ROOT.parent if ROOT.name == "notebooks" else ROOT
sys.path.insert(0, str(ROOT))
print("Project root:", ROOT)

In [ ]:
# ---- configuration -------------------------------------------------------
import torch
VIDEO      = "street"            # "people", "cars" or "street"
EVERY_S    = 1.0                 # keyframe spacing in seconds
CLIP_ID    = "openai/clip-vit-base-patch32"
GPU_GB     = torch.cuda.get_device_properties(0).total_memory / 1e9 if torch.cuda.is_available() else 0
VLM_ID     = ("HuggingFaceTB/SmolVLM2-2.2B-Instruct" if GPU_GB >= 7     # needs ~5 GB in 16-bit
              else "HuggingFaceTB/SmolVLM2-500M-Video-Instruct")      # fits 4 GB GPUs
SPLIT_IMAGES = False             # True: VLM tiles each frame (more detail, ~10x slower); frames are only 512 px
DETECTOR   = "yolo11s.pt"
TRACK_CLASSES = {0: "person", 1: "bicycle", 2: "car", 3: "motorcycle", 5: "bus", 7: "truck"}   # COCO ids
N_VLM_FRAMES     = 10            # keyframes the VLM answers questions about, one at a time
N_SUMMARY_FRAMES = 6             # keyframes sent together for the whole-clip summary
QUERIES = ["a person walking", "a person riding a bicycle", "a car driving on the road",
           "a group of people", "an empty street"]

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from PIL import Image
from IPython.display import Image as IPImage, display
from ultralytics import YOLO
from vlmkit.common import get_device, out_dir, save_json, show_grid, live, Timer
from vlmkit.clip_utils import Clip
from vlmkit.vlm import SmallVLM, parse_int, parse_yes_no
from vlmkit import data

DEVICE, OUT = get_device(), out_dir("c5")
videos = data.videos()
for name, p in videos.items():
    i = data.video_info(p)
    print(f"{name:7s} {p.name:36s} {i['seconds']:5.1f} s  {i['fps']:.1f} fps  {i['width']}x{i['height']}")
path, info = videos[VIDEO], data.video_info(videos[VIDEO])
keys = data.extract_keyframes(path, every_s=EVERY_S)
print(f"\nusing '{VIDEO}': {len(keys)} keyframes, one every {EVERY_S:g} s")
step = max(len(keys) // 10, 1)
show_grid([Image.open(k["path"]) for k in keys[::step][:10]], [f"t = {k['t']:.0f} s" for k in keys[::step][:10]],
          cols=5, size=2.8, save=OUT / f"{VIDEO}_keyframes.png")

## 1. Baseline: detector + tracker on every frame

A tracker gives each object an ID that persists across frames, so we can count *unique* people and cars, not just boxes per frame.
This is the reference we will check the language models against (it makes mistakes too).

In [ ]:
detector = YOLO(DETECTOR)
rows, preview = [], []
gif_every = max(info["frames"] // 80, 1)                     # at most ~80 frames in the preview GIF
with Timer() as t:
    for i, r in enumerate(detector.track(str(path), stream=True, persist=True, classes=list(TRACK_CLASSES),
                                         conf=0.3, tracker="bytetrack.yaml", verbose=False)):
        cls = r.boxes.cls.cpu().numpy().astype(int)
        ids = r.boxes.id.cpu().numpy().astype(int) if r.boxes.id is not None else np.full(len(cls), -1)
        rows += [{"frame": i, "t": i / info["fps"], "class": TRACK_CLASSES[c], "track_id": d} for c, d in zip(cls, ids)]
        if i % gif_every == 0:
            im = Image.fromarray(r.plot()[:, :, ::-1]); im.thumbnail((480, 480)); preview.append(im)
n_frames = i + 1
print(f"tracked {n_frames} frames in {t.seconds:.1f} s ({n_frames / t.seconds:.0f} frames/s)")
det = pd.DataFrame(rows, columns=["frame", "t", "class", "track_id"])
det.to_csv(OUT / f"{VIDEO}_tracks.csv", index=False)
per_frame = det.groupby(["frame", "class"]).size().unstack(fill_value=0).reindex(range(n_frames), fill_value=0)
unique = det[det.track_id >= 0].groupby("class").track_id.nunique()
pd.DataFrame({"unique objects (tracker)": unique, "max in one frame": per_frame.max()})

### Watch it: tracked video (plays in the notebook)

In [ ]:
gif = OUT / f"{VIDEO}_tracked.gif"
preview[0].save(gif, save_all=True, append_images=preview[1:], duration=int(1000 * gif_every / info["fps"]), loop=0)
display(IPImage(filename=str(gif)))

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.2))
for c in per_frame.columns:
    ax.plot(per_frame.index / info["fps"], per_frame[c], label=c, lw=1.2)
ax.set_xlabel("time (s)"); ax.set_ylabel("objects in frame"); ax.legend(ncol=3)
ax.set_title(f"{VIDEO}: detections per frame (YOLO11s + ByteTrack)"); plt.tight_layout()
plt.savefig(OUT / f"{VIDEO}_counts_over_time.png", dpi=120); plt.show()

## 2. Search the video with sentences (CLIP)

Embed every keyframe once; each sentence becomes a similarity curve over time. Bright = good match.

In [ ]:
clip = Clip(CLIP_ID, DEVICE)
key_imgs = [Image.open(k["path"]).convert("RGB") for k in keys]
KV = clip.images(key_imgs).numpy()
sims = clip.texts(QUERIES).numpy() @ KV.T                    # (queries, keyframes)

fig, ax = plt.subplots(figsize=(9, 0.45 * len(QUERIES) + 1.6))
im = ax.imshow(sims, aspect="auto", cmap="viridis",
               extent=[keys[0]["t"], keys[-1]["t"] + EVERY_S, len(QUERIES) - 0.5, -0.5])
ax.set_yticks(range(len(QUERIES))); ax.set_yticklabels(QUERIES); ax.set_xlabel("time (s)")
fig.colorbar(im, ax=ax, label="cosine similarity"); ax.set_title("When does each sentence match the video?")
plt.tight_layout(); plt.savefig(OUT / f"{VIDEO}_clip_timeline.png", dpi=120); plt.show()

best = sims.argmax(1)
show_grid([key_imgs[j] for j in best], [f"{q}\nt = {keys[j]['t']:.0f} s" for q, j in zip(QUERIES, best)],
          cols=len(QUERIES), size=2.8, save=OUT / f"{VIDEO}_clip_best_frames.png")

## 3. Ask the VLM about single frames, check against the detector

For each sampled keyframe: a one-sentence caption, two counting questions, two yes/no questions.
The detector's count on the same frame is the reference.

In [ ]:
with Timer() as t:
    vlm = SmallVLM(VLM_ID, device="cuda" if DEVICE == "cuda" else "cpu", split_images=SPLIT_IMAGES)
print(f"loaded {VLM_ID} on {vlm.device} ({vlm.dtype}) in {t.seconds:.0f} s")

def det_count(frame, cls):
    return int(per_frame.loc[frame, cls]) if cls in per_frame.columns else 0

pick = [keys[j] for j in np.linspace(0, len(keys) - 1, min(N_VLM_FRAMES, len(keys))).astype(int)]
rows = []
for k in pick:
    p = k["path"]
    with Timer() as tm:
        caption = vlm.ask(p, "Describe this video frame in one sentence.")
        n_people = parse_int(vlm.ask(p, "How many people are in this image? Answer with a number only.", max_new_tokens=8))
        n_cars = parse_int(vlm.ask(p, "How many cars are in this image? Answer with a number only.", max_new_tokens=8))
        says_person = parse_yes_no(vlm.ask(p, "Is there a person in this image? Answer yes or no.", max_new_tokens=4))
        says_car = parse_yes_no(vlm.ask(p, "Is there a car in this image? Answer yes or no.", max_new_tokens=4))
    rows.append({"t": k["t"], "frame": k["frame"], "caption": caption,
                 "vlm_people": n_people, "det_people": det_count(k["frame"], "person"),
                 "vlm_cars": n_cars, "det_cars": det_count(k["frame"], "car"),
                 "vlm_says_person": says_person, "vlm_says_car": says_car, "seconds": round(tm.seconds, 1)})
fr = pd.DataFrame(rows); fr.to_csv(OUT / f"{VIDEO}_vlm_frames.csv", index=False)
fr[["t", "caption", "vlm_people", "det_people", "vlm_cars", "det_cars", "seconds"]]

In [ ]:
def agree(obj):
    v, d = pd.to_numeric(fr[f"vlm_{obj}"], errors="coerce"), fr[f"det_{obj}"]
    ok = v.notna(); err = (v[ok] - d[ok]).abs()
    return {"answered": f"{ok.mean():.0%}", "exact vs detector": round(float((err == 0).mean()), 3),
            "MAE vs detector": round(float(err.mean()), 2)}

def presence(obj, det_col):
    v = fr[f"vlm_says_{obj}"]; ok = v.notna(); seen = fr[det_col] > 0
    return {"VLM yes, detector none": int(((v == True) & ~seen).sum()),
            "VLM no, detector found": int(((v == False) & seen).sum()),
            "agreement": round(float(((v == True) == seen)[ok].mean()), 3) if ok.any() else None}

counting = pd.DataFrame({"people": agree("people"), "cars": agree("cars")}).T
yes_no = pd.DataFrame({"person": presence("person", "det_people"), "car": presence("car", "det_cars")}).T
display(counting, yes_no)

fig, axes = plt.subplots(1, 2, figsize=(10, 3.2), sharex=True)
for ax, obj in zip(axes, ["people", "cars"]):
    ax.plot(fr.t, fr[f"det_{obj}"], "s-", label="detector", alpha=0.8)
    ax.plot(fr.t, pd.to_numeric(fr[f"vlm_{obj}"], errors="coerce"), "o--", label="VLM", alpha=0.8)
    ax.set_title(f"{obj} per frame"); ax.set_xlabel("time (s)"); ax.legend()
plt.tight_layout(); plt.savefig(OUT / f"{VIDEO}_vlm_vs_detector.png", dpi=120); plt.show()

show_grid([Image.open(k["path"]) for k in pick[:6]],
          [f"t={r.t:.0f}s  VLM {r.vlm_people}p/{r.vlm_cars}c  det {r.det_people}p/{r.det_cars}c\n{r.caption[:70]}"
           for r in fr.head(6).itertuples()], cols=3, size=3.6, save=OUT / f"{VIDEO}_vlm_frames.png")

## 4. One question about the whole clip

Several keyframes go into a single prompt, in time order. Compare the story with the tracker's facts: does the model invent people, vehicles or actions?

In [ ]:
summary_keys = [keys[j] for j in np.linspace(0, len(keys) - 1, min(N_SUMMARY_FRAMES, len(keys))).astype(int)]
gap = (summary_keys[-1]["t"] - summary_keys[0]["t"]) / max(len(summary_keys) - 1, 1)
prompt = (f"These {len(summary_keys)} images are frames from one video, in time order, about {gap:.0f} seconds apart. "
          "Describe what happens in the video: the people and vehicles you see and how they move.")
with Timer() as tm:
    story = vlm.ask([k["path"] for k in summary_keys], prompt, max_new_tokens=160)
show_grid([Image.open(k["path"]) for k in summary_keys], [f"t = {k['t']:.0f} s" for k in summary_keys],
          cols=len(summary_keys), size=2.4)
print(f"VLM summary ({tm.seconds:.1f} s):\n{story}\n")
print("Tracker facts to check it against (unique objects):", unique.to_dict())

## 5. Try it live

**Frame explorer:** pick a moment in the video, see the detector's boxes and ask the VLM anything about that frame.

In [ ]:
frame_detector = YOLO(DETECTOR)          # separate copy: the tracking model keeps tracker state

def frame_panel(keyframe, question, show_boxes):
    k = keys[keyframe]
    if show_boxes:
        r = frame_detector.predict(str(k["path"]), classes=list(TRACK_CLASSES), conf=0.3, verbose=False)[0]
        img = Image.fromarray(r.plot()[:, :, ::-1])
    else:
        img = Image.open(k["path"])
    show_grid([img], [f"{VIDEO}  t = {k['t']:.0f} s"], cols=1, size=6)
    if question.strip():
        with Timer() as tm:
            answer = vlm.ask(k["path"], question, max_new_tokens=96)
        print(f"Q: {question}\nA: {answer}   ({tm.seconds:.1f} s)")

live(frame_panel, keyframe=(0, len(keys) - 1, 1, len(keys) // 2),
     question="What are the people doing, and where are they going?", show_boxes=True)

**Video search:** choose any of the three videos and type a sentence; CLIP returns the best-matching moments.

In [ ]:
_video_cache = {}

def video_search_panel(video, query, top_k):
    if video not in _video_cache:
        ks = data.extract_keyframes(videos[video], every_s=EVERY_S)
        ims = [Image.open(k["path"]).convert("RGB") for k in ks]
        _video_cache[video] = (ks, ims, clip.images(ims).numpy())
    ks, ims, E = _video_cache[video]
    s = E @ clip.texts([query]).numpy()[0]
    top = np.argsort(-s)[:top_k]
    show_grid([ims[j] for j in top], [f"t = {ks[j]['t']:.0f} s  ({s[j]:.3f})" for j in top],
              cols=min(top_k, 4), size=3)

live(video_search_panel, video=list(videos), query="a person carrying a bag", top_k=(1, 8, 1, 4))

In [ ]:
summary = {
    "video": VIDEO, "file": path.name, "seconds": round(info["seconds"], 1), "keyframes": len(keys),
    "vlm": VLM_ID, "clip": CLIP_ID, "detector": DETECTOR,
    "tracker_unique_objects": {k: int(v) for k, v in unique.items()},
    "clip_best_time_per_query": {q: keys[j]["t"] for q, j in zip(QUERIES, best)},
    "vlm_counting_vs_detector": counting.to_dict("index"),
    "vlm_yes_no_vs_detector": yes_no.to_dict("index"),
    "vlm_seconds_per_frame": round(float(fr.seconds.mean()), 2),
    "vlm_clip_summary": story,
}
save_json(summary, OUT / f"{VIDEO}_summary.json")
summary

## Exercises for students

1. Switch `VIDEO` to `"people"` and `"cars"` and re-run. On which video does the VLM count best, and why (size of objects, crowding, occlusion)?
2. Halve `EVERY_S`. Does the CLIP timeline find short events (a cyclist passing) that 1-second keyframes miss?
3. Change the summary prompt to ask for a *list of events with timestamps*. How often are the timestamps right according to the tracker plot?
4. **Transfer to your project:** point `data.extract_keyframes` at one of your own traffic or CCTV clips and use section 3 as a guardrail: flag frames where the VLM and the detector disagree.